# X_EXTRAS

This notebook generates a LaTeX table of observables used in the sweep tests. It reads `dd` from `recipe.py`, loads the sweep parameter JSON files, and writes the final table to `output/tex/observables_table.tex`.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path



In [3]:
#!/usr/bin/env python3
from __future__ import annotations

import importlib.util
import json
from pathlib import Path
from typing import Any


SWEEP_DIR = Path("output/sweeps")
RECIPE_PATH = Path("recipe.py")
OUT_PATH = Path("output/tex/observables_table.tex")

JSON_FILES = {
    "GBM": SWEEP_DIR / "gbm_params.json",
    "QRF": SWEEP_DIR / "qrf_params.json",
    "SIM": SWEEP_DIR / "sim_best_params.json",
}

MODEL_COLUMNS = ["GBM", "QRF", "SIM"]

FORWARD_EXTRA_ROWS = [
    {
        "label": "VOLCANOES",
        "description": "Point observation (Aq2 only)",
        "reference": "GVP",
        "grid": "Forward",
    }, 
        {
        "label": "HYDROTHERMAL",
        "description": "Point observation (Kq2 only)",
        "reference": "GVP",
        "grid": "Forward",
    }
]


FORWARD_LABELS = {row["label"] for row in FORWARD_EXTRA_ROWS}



def load_recipe_dd(recipe_path: Path) -> list[dict[str, Any]]:
    if not recipe_path.exists():
        raise FileNotFoundError(f"Missing recipe file: {recipe_path}")

    spec = importlib.util.spec_from_file_location("aq2_recipe", recipe_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Could not import recipe file: {recipe_path}")

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, "dd"):
        raise AttributeError(f"{recipe_path} does not define a variable named 'dd'")

    return module.dd


def load_obs_sel(path: Path) -> set[str]:
    if not path.exists():
        raise FileNotFoundError(f"Missing sweep JSON file: {path}")

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if "obs_sel" not in data:
        raise KeyError(f"No 'obs_sel' key found in {path}")

    return set(str(label) for label in data["obs_sel"])


def hashable_grid(grid: Any) -> Any:
    if isinstance(grid, list):
        return tuple(grid)
    if isinstance(grid, set):
        return tuple(sorted(grid))
    return grid


def latex_escape_text(text: str) -> str:
    if not text:
        return ""

    protected = {}
    for i, token in enumerate(["\\citet", "\\citep", "\\cite", "\\url"]):
        key = f"@@LATEXCMD{i}@@"
        protected[key] = token
        text = text.replace(token, key)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    escaped = "".join(replacements.get(ch, ch) for ch in text)

    for key, token in protected.items():
        escaped = escaped.replace(key, token)

    escaped = (
        escaped
        .replace(r"\citet\{", r"\citet{")
        .replace(r"\citep\{", r"\citep{")
        .replace(r"\cite\{", r"\cite{")
        .replace(r"\url\{", r"\url{")
        .replace(r"\}", "}")
    )

    return escaped


def latex_label(label: str) -> str:
    return label.replace("_", r"\_")


def latex_reference(reference: str | None) -> str:
    if not reference:
        return ""

    reference = str(reference).strip()
    if not reference:
        return ""

    if reference.startswith("\\"):
        return reference

    if "/" in reference or reference.startswith("10."):
        return latex_escape_text(reference)

    return rf"\citet{{{reference}}}"


def bool_macro(value: bool) -> str:
    return r"\boolTrue" if value else r"\boolFalse"


def grid_matches(grid_value: Any, wanted: str) -> bool:
    if isinstance(grid_value, str):
        return grid_value == wanted
    if isinstance(grid_value, list | tuple | set):
        return wanted in grid_value
    return False


def is_forward_true(row: dict[str, Any]) -> bool:
    label = row.get("label")
    grid = row.get("grid")

    if label in FORWARD_LABELS:
        return True

    if label == "DEM" and (
        grid_matches(grid, "Antarctica") or grid_matches(grid, "Greenland")
    ):
        return True

    return False


def recipe_rows_for_selected_labels(
    dd: list[dict[str, Any]],
    selected_labels: set[str],
) -> list[dict[str, Any]]:
    rows = []

    for entry in dd:
        if not isinstance(entry, dict):
            continue

        label = entry.get("label")
        if label in selected_labels:
            rows.append(entry)

    return rows


def add_missing_label_rows(
    rows: list[dict[str, Any]],
    selected_labels: set[str],
) -> list[dict[str, Any]]:
    present = {row.get("label") for row in rows}

    for label in sorted(selected_labels - present):
        rows.append(
            {
                "label": label,
                "description": label,
                "reference": "",
                "grid": "",
            }
        )

    return rows


def main() -> None:
    obs_selected = {
        model: load_obs_sel(path)
        for model, path in JSON_FILES.items()
    }

    # Labels selected by the ML models only.
    json_labels = set().union(*obs_selected.values())

    dd = load_recipe_dd(RECIPE_PATH)

    # Get rows from recipe.py only for labels found in the JSON obs_sel lists.
    rows = recipe_rows_for_selected_labels(dd, json_labels)

    # Add fallback rows only for JSON labels missing from recipe.py.
    # Do NOT include forward-only labels here, otherwise VOLCANOES etc.
    # get added once as fallback rows and then again as explicit forward rows.
    rows = add_missing_label_rows(rows, json_labels)

    existing_pairs = {
        (
            row.get("label"),
            hashable_grid(row.get("grid")),
            row.get("description"),
        )
        for row in rows
    }

    # Append explicit forward-only rows.
    for row in FORWARD_EXTRA_ROWS:
        key = (
            row.get("label"),
            hashable_grid(row.get("grid")),
            row.get("description"),
        )
        if key not in existing_pairs:
            rows.append(row)
            existing_pairs.add(key)

    lines = [
        r"\begin{tabular}{p{3cm} p{6.3cm} C{0.5cm} C{0.5cm} C{0.5cm} C{0.5cm} p{3.5cm}}",
        "",
        r"    Label & Description & GBM & QRF & SIM & F. & \\",
        r"    \midrule",
    ]

    for row in rows:
        label = str(row.get("label", ""))
        description = str(row.get("description", label))
        reference = latex_reference(row.get("reference"))

        model_flags = [
            bool_macro(label in obs_selected[model])
            for model in MODEL_COLUMNS
        ]

        forward_flag = bool_macro(is_forward_true(row))

        lines.append(
            f"    {latex_label(label)} & "
            f"{latex_escape_text(description)} & "
            f"{model_flags[0]} & {model_flags[1]} & {model_flags[2]} & "
            f"{forward_flag} & "
            f"{reference} \\\\"
        )

    lines.extend(
        [
            r"    \bottomrule",
            r"\end{tabular}",
            "",
        ]
    )

    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUT_PATH.write_text("\n".join(lines), encoding="utf-8")

    print(f"Wrote {OUT_PATH}")


if __name__ == "__main__":
    main()

Wrote output/tex/observables_table.tex
